# Acute Ischemic Stroke Risk Classification Engine - Phase 1

## Dataset Exploration and Exploratory Data Analysis

This notebook explores the Kaggle stroke prediction dataset, checks data quality, studies feature distributions, evaluates correlations, and documents the main insights required before moving to model development.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'Stroke prediction dataset').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.eda_utils import (
    create_bivariate_plots,
    create_categorical_countplots,
    create_correlation_heatmap,
    create_distribution_analysis_plots,
    create_missing_values_plot,
    create_numeric_distribution_plots,
    create_outlier_plots,
    create_target_plots,
    dataframe_profile,
    dataset_quality_notes,
    ensure_output_directories,
    load_dataset,
    missing_summary,
    normality_summary,
    outlier_summary,
    statistical_summary,
    target_distribution,
    target_percentages,
)

ensure_output_directories()
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# Dataset Loading

The dataset is loaded automatically by searching for `healthcare-dataset-stroke-data.csv` inside the workspace. The next cell shows the first and last rows, dataset shape, column names, data types, and memory usage.

In [2]:
df = load_dataset(PROJECT_ROOT)

print('First 5 rows:')
display(df.head())

print('Last 5 rows:')
display(df.tail())

print('Dataset shape:', df.shape)
print('Column names:', list(df.columns))
print('Data types:')
display(df.dtypes.to_frame(name='dtype'))
print(f'Memory usage: {df.memory_usage(deep=True).sum() / (1024 ** 2):.2f} MB')

First 5 rows:


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


Last 5 rows:


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
5105,18234,Female,80.0,1,0,Yes,Private,Urban,83.75,NaN,never smoked,0
5106,44873,Female,81.0,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35.0,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51.0,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0
5109,44679,Female,44.0,0,0,Yes,Govt_job,Urban,85.28,26.2,Unknown,0


Dataset shape: (5110, 12)
Column names: ['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']
Data types:


,dtype
id,int64
gender,object
age,float64
hypertension,int64
heart_disease,int64
ever_married,object
work_type,object
Residence_type,object
avg_glucose_level,float64
bmi,float64


Memory usage: 1.62 MB


# Dataset Overview

This section uses `info()`, `describe()`, and `describe(include="object")` to summarize the dataset structure and the distribution of both numerical and categorical fields.

In [3]:
df.info()

print('\nNumerical summary:')
display(df.describe())

print('Categorical summary:')
display(df.describe(include='object'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB

Numerical summary:


,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


Categorical summary:


,gender,ever_married,work_type,Residence_type,smoking_status
count,5110,5110,5110,5110,5110
unique,3,2,5,2,4
top,Female,Yes,Private,Urban,never smoked
freq,2994,3353,2925,2596,1892


# Data Quality Analysis

This section checks missing values, duplicate rows, unique values in categorical columns, and any obvious invalid or rare entries. The missing-values plot is also saved into the figures folder.

In [4]:
profile = dataframe_profile(df)
missing_df = missing_summary(df)

display(missing_df)
print('Total missing values:', int(profile['missing_values'].sum()))
print('Duplicate rows:', profile['duplicate_rows'])

categorical_columns = df.select_dtypes(include='object').columns.tolist()
for column in categorical_columns:
    print(f'\nUnique values for {column}:')
    display(df[column].value_counts(dropna=False).to_frame(name='count'))

print('Quality notes:')
for note in dataset_quality_notes(df):
    print('-', note)

create_missing_values_plot(df)

,missing_values,missing_percentage
bmi,201,3.93
id,0,0.00
age,0,0.00
gender,0,0.00
hypertension,0,0.00
heart_disease,0,0.00
work_type,0,0.00
ever_married,0,0.00
Residence_type,0,0.00
avg_glucose_level,0,0.00


Total missing values: 201
Duplicate rows: 0

Unique values for gender:


,count
gender,
Female,2994
Male,2115
Other,1



Unique values for ever_married:


,count
ever_married,
Yes,3353
No,1757


...

,count
Residence_type,
Urban,2596
Rural,2514



Unique values for smoking_status:


,count
smoking_status,
never smoked,1892
Unknown,1544
formerly smoked,885
smokes,789


Quality notes:
- BMI contains missing values and should be imputed before model training.
- Gender contains a rare 'Other' category with a single record.
- Never_worked is a very small category and may need grouping in modeling.
- No duplicate rows were found.


C:\Users\gupta\OneDrive\Attachments\Desktop\ML project\src\eda_utils.py:333: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=summary.index.astype(str), y=summary["missing_percentage"].values, palette="magma")


# Target Variable Analysis

The target variable is `stroke`. The next cell creates the count plot, pie chart, and percentage distribution to show the class imbalance clearly.

In [5]:
target_df = pd.DataFrame({
    'count': target_distribution(df),
    'percentage': target_percentages(df),
})
display(target_df)
create_target_plots(df)

,count,percentage
stroke,,
0,4861,95.13
1,249,4.87


C:\Users\gupta\OneDrive\Attachments\Desktop\ML project\src\eda_utils.py:169: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countplot(x=TARGET_COLUMN, data=df, palette="Set2")


C:\Users\gupta\OneDrive\Attachments\Desktop\ML project\src\eda_utils.py:191: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=percentages.index.astype(str), y=percentages.values, palette="Set1")


The target distribution is highly imbalanced. The non-stroke class dominates the sample, so accuracy alone would be misleading during modeling. Recall, precision, F1-score, and ROC-AUC will matter more later.

# Numerical Features

The numerical columns `age`, `bmi`, and `avg_glucose_level` are studied with histograms, KDE curves, boxplots, and violin plots. These views help identify skewness, spread, and potential outliers.

In [6]:
display(statistical_summary(df))
create_numeric_distribution_plots(df)

,mean,median,mode,variance,std_dev,skewness,kurtosis
age,43.2266,45.000,78.00,511.3318,22.6126,-0.1371,-0.9910
bmi,28.8932,28.100,28.70,61.6864,7.8541,1.0553,3.3627
avg_glucose_level,106.1477,91.885,93.88,2050.6008,45.2836,1.5723,1.6805


`age` is relatively balanced and close to symmetric, while `avg_glucose_level` and `bmi` are right-skewed. The boxplots and violin plots show that glucose has the heaviest upper tail, and BMI has moderate spread with missing values to handle later.

# Categorical Features

The following cell creates count plots for all categorical variables and prints their frequency tables so the class balance inside each category is easy to inspect.

In [7]:
create_categorical_countplots(df)
for column in ['gender', 'work_type', 'ever_married', 'Residence_type', 'smoking_status', 'hypertension', 'heart_disease']:
    print(f'\n{column}:')
    display(df[column].value_counts(dropna=False).to_frame(name='count'))

C:\Users\gupta\OneDrive\Attachments\Desktop\ML project\src\eda_utils.py:239: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countplot(x=column, data=df, order=order, palette="viridis")


C:\Users\gupta\OneDrive\Attachments\Desktop\ML project\src\eda_utils.py:239: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countplot(x=column, data=df, order=order, palette="viridis")


C:\Users\gupta\OneDrive\Attachments\Desktop\ML project\src\eda_utils.py:239: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countplot(x=column, data=df, order=order, palette="viridis")


C:\Users\gupta\OneDrive\Attachments\Desktop\ML project\src\eda_utils.py:239: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countplot(x=column, data=df, order=order, palette="viridis")


C:\Users\gupta\OneDrive\Attachments\Desktop\ML project\src\eda_utils.py:239: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countplot(x=column, data=df, order=order, palette="viridis")


...

,count
smoking_status,
never smoked,1892
Unknown,1544
formerly smoked,885
smokes,789



hypertension:


,count
hypertension,
0,4612
1,498



heart_disease:


,count
heart_disease,
0,4834
1,276


The categorical fields are mostly clean. `gender` contains a single `Other` record, `Never_worked` is very rare, and `Unknown` appears frequently in smoking status. These categories may need special treatment during preprocessing.

# Bivariate Analysis

This section compares stroke outcomes against selected demographic, clinical, and lifestyle variables. The saved plots show how stroke risk changes across groups.

In [8]:
create_bivariate_plots(df)

C:\Users\gupta\OneDrive\Attachments\Desktop\ML project\src\eda_utils.py:268: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(x=TARGET_COLUMN, y=column, data=df, palette="Set3")


C:\Users\gupta\OneDrive\Attachments\Desktop\ML project\src\eda_utils.py:268: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(x=TARGET_COLUMN, y=column, data=df, palette="Set3")


C:\Users\gupta\OneDrive\Attachments\Desktop\ML project\src\eda_utils.py:268: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(x=TARGET_COLUMN, y=column, data=df, palette="Set3")


Stroke appears more frequently among older patients, patients with hypertension, patients with heart disease, married patients, and self-employed patients. The plots also suggest that former smokers show higher stroke proportions than most other smoking groups.

# Correlation Analysis

Categorical features are encoded and then used to generate a correlation matrix and heatmap. This helps identify the strongest positive and negative linear relationships with the target.

In [9]:
corr = create_correlation_heatmap(df)
display(corr['stroke'].sort_values(ascending=False).to_frame(name='correlation_to_stroke'))

,correlation_to_stroke
stroke,1.000000
age,0.245257
heart_disease,0.134914
avg_glucose_level,0.131945
hypertension,0.127904
ever_married_Yes,0.108340
smoking_status_formerly smoked,0.064556
work_type_Self-employed,0.062168
bmi,0.042374
Residence_type_Urban,0.015458


Age is the strongest positive feature related to stroke in this encoded correlation view. Heart disease, glucose level, hypertension, and being married also show positive association, while children, unmarried status, and unknown smoking status show negative association.

# Outlier Detection

Outlier analysis is performed on the three numerical features using IQR-based boxplots and a compact summary table.

In [10]:
display(outlier_summary(df))
create_outlier_plots(df)

,feature,lower_bound,upper_bound,outlier_count,outlier_percentage
0,age,-29.0000,115.0000,0,0.00
1,bmi,9.1000,47.5000,110,2.15
2,avg_glucose_level,21.9775,169.3575,627,12.27


`avg_glucose_level` shows the most visible outlier behavior, `bmi` has moderate outlier activity, and `age` remains well-behaved under the IQR rule.

# Distribution Analysis

QQ plots and a normality test are used to assess whether the numerical features follow a normal distribution.

In [11]:
display(normality_summary(df))
create_distribution_analysis_plots(df)

,feature,normaltest_statistic,p_value,normal_at_0.05
0,age,1120.528644,4.789570e-244,False
1,bmi,1021.179505,1.793444e-222,False
2,avg_glucose_level,1328.935795,2.662311e-289,False


The QQ plots and normality results indicate that the numerical variables are not normally distributed, especially `avg_glucose_level` and `bmi`, which are clearly right-skewed.

# Conclusions

The dataset is clean overall, but it is highly imbalanced and contains missing BMI values, skewed numerical variables, and a few rare categorical entries. These findings point to the need for imputation, encoding, scaling, and imbalance handling before training a classification model.